# Rock vs Mine Prediction using SONAR Data
> **Binary Classification with Logistic Regression**

| Attribute | Detail |
|-----------|--------|
| **Dataset** | UCI SONAR Dataset — 208 samples, 60 frequency-band features |
| **Task** | Classify sonar signals as Rock (R) or Mine (M) |
| **Algorithm** | Logistic Regression (scikit-learn) |
| **Best Test Accuracy** | ~76% |

---


## 1. Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# Reproducibility
RANDOM_STATE = 42
print("Libraries loaded successfully.")


## 2. Load & Inspect the Dataset

In [ ]:
# The SONAR dataset has no header row; column 60 is the label (R = Rock, M = Mine)
sonar_data = pd.read_csv('sonar data.csv', header=None)

print(f"Shape: {sonar_data.shape}")
sonar_data.head()


## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
print("Class distribution:")
print(sonar_data[60].value_counts())
print()

# Statistical summary of first 5 features
print("Statistical summary (first 5 features):")
sonar_data.iloc[:, :5].describe().round(4)


In [ ]:
# Mean feature values per class
class_means = sonar_data.groupby(60).mean()
print("Mean feature values by class (first 5 features):")
class_means.iloc[:, :5].round(4)


In [ ]:
# Visualise mean signal per class across all 60 bands
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(class_means.loc['R'].values, label='Rock', color='#58A6FF', linewidth=2)
ax.plot(class_means.loc['M'].values, label='Mine', color='#FF7B00', linewidth=2)
ax.fill_between(range(60), class_means.loc['R'].values, alpha=0.15, color='#58A6FF')
ax.fill_between(range(60), class_means.loc['M'].values, alpha=0.15, color='#FF7B00')
ax.set(title='Mean Sonar Energy per Frequency Band — Rock vs Mine',
       xlabel='Frequency Band', ylabel='Mean Energy')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Data Preparation

In [ ]:
X = sonar_data.drop(columns=60)   # features (60 sonar bands)
y = sonar_data[60]                # labels  (R / M)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,            # keep class balance in both splits
    random_state=RANDOM_STATE
)

print(f"Total samples  : {X.shape[0]}")
print(f"Training set   : {X_train.shape[0]} samples")
print(f"Test set       : {X_test.shape[0]} samples")
print(f"Features       : {X.shape[1]}")


## 5. Model Training — Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print("Model training complete.")


## 6. Model Evaluation

In [ ]:
train_preds = model.predict(X_train)
test_preds  = model.predict(X_test)

train_acc = accuracy_score(y_train, train_preds)
test_acc  = accuracy_score(y_test,  test_preds)

print(f"Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Test Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)")


In [ ]:
# Full classification report
print("Classification Report (Test Set):")
print(classification_report(y_test, test_preds, target_names=['Rock', 'Mine']))


In [ ]:
# Confusion matrix
cm  = confusion_matrix(y_test, test_preds, labels=['R', 'M'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart — accuracy comparison
axes[0].barh(['Training', 'Test'], [train_acc * 100, test_acc * 100],
             color=['#39D353', '#58A6FF'], height=0.45)
axes[0].set_xlim(0, 105)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Training vs Test Accuracy')
for i, v in enumerate([train_acc * 100, test_acc * 100]):
    axes[0].text(v + 1, i, f'{v:.2f}%', va='center', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Confusion matrix heat-map
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rock', 'Mine'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (Test Set)')

plt.tight_layout()
plt.show()


## 7. Predictive System

A reusable function that classifies a single 60-band sonar reading.


In [ ]:
def predict_object(sonar_readings: list) -> str:
    """
    Classify a sonar signal as Rock or Mine.

    Parameters
    ----------
    sonar_readings : list of 60 float values
        Energy values for frequency bands 1-60.

    Returns
    -------
    str : 'Rock' or 'Mine'
    """
    if len(sonar_readings) != 60:
        raise ValueError(f"Expected 60 features, got {len(sonar_readings)}.")

    arr  = np.asarray(sonar_readings).reshape(1, -1)
    pred = model.predict(arr)[0]
    prob = model.predict_proba(arr)[0]

    label    = "Rock" if pred == "R" else "Mine"
    conf_idx = 0 if pred == "R" else 1
    confidence = prob[conf_idx] * 100

    print(f"Prediction  : {label}")
    print(f"Confidence  : {confidence:.1f}%")
    return label


# ── Sample prediction ──────────────────────────────────────────────────────
sample_input = (
    0.0307, 0.0523, 0.0653, 0.0521, 0.0611, 0.0577, 0.0665, 0.0664,
    0.1460, 0.2792, 0.3877, 0.4992, 0.4981, 0.4972, 0.5607, 0.7339,
    0.8230, 0.9173, 0.9975, 0.9911, 0.8240, 0.6498, 0.5980, 0.4862,
    0.3150, 0.1543, 0.0989, 0.0284, 0.1008, 0.2636, 0.2694, 0.2930,
    0.2925, 0.3998, 0.3660, 0.3172, 0.4609, 0.4374, 0.1820, 0.3376,
    0.6202, 0.4448, 0.1863, 0.1420, 0.0589, 0.0576, 0.0672, 0.0269,
    0.0245, 0.0190, 0.0063, 0.0321, 0.0189, 0.0137, 0.0277, 0.0152,
    0.0052, 0.0121, 0.0124, 0.0055
)

result = predict_object(list(sample_input))


## 8. Key Takeaways

| Insight | Detail |
|---------|--------|
| **Algorithm choice** | Logistic Regression works well as a strong baseline for binary classification |
| **Feature space** | 60 sonar frequency bands provide rich signal; no dimensionality reduction needed |
| **Train-test gap** | ~7% gap suggests mild overfitting — tuning `C` or adding regularisation can help |
| **Next steps** | Try SVM, Random Forest, or a shallow MLP; apply StandardScaler before training |

---

### References
- Dataset: [UCI Machine Learning Repository — Connectionist Bench (Sonar)](https://archive.ics.uci.edu/ml/datasets/connectionist+bench+(sonar,+mines+vs.+rocks))
- Gorman & Sejnowski (1988), *Analysis of Hidden Units in a Layered Network Trained to Classify Sonar Targets*
